# Étape 1 (suite) : anonymisation avec Presidio

Entrée : les fichiers JSONL du notebook 01 (`data/interim/`). Sortie : les mêmes fichiers anonymisés (`data/processed/`), avec le nombre d'éléments masqués dans chaque exemple.

Ce qu'on masque : les identifiants directs, c'est-à-dire les noms et prénoms, les adresses e-mail et les numéros de téléphone. On garde l'âge, le sexe, les lieux et les dates : seuls, ils ne permettent pas de retrouver une personne dans ces données, et ils comptent pour le raisonnement médical (un séjour au Gabon oriente vers le paludisme). C'est le principe de minimisation du RGPD : on retire ce qui identifie une personne et on garde ce qui sert au raisonnement.

Les sources ne sont pas toutes au même niveau de risque. Les QCM d'examens et les pages des NIH sont des textes rédigés, avec des patients fictifs. Les questions d'UltraMedical qui viennent de ChatDoctor sont de vrais messages de patients postés en ligne : c'est là qu'on s'attend à trouver de vraies données personnelles.

In [1]:
import csv
import json
import random
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from faker.providers.person.en_GB import Provider as NamesGB
from faker.providers.person.en_IN import Provider as NamesIN
from faker.providers.person.en_US import Provider as NamesUS
from faker.providers.person.fr_FR import Provider as NamesFR
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

SEED = 42
IN_DIR = Path("../data/interim")
OUT_DIR = Path("../data/processed")
CTRL_DIR = Path("../data/controle")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CTRL_DIR.mkdir(parents=True, exist_ok=True)

files = {p.stem: [json.loads(line) for line in open(p, encoding="utf-8")] for p in sorted(IN_DIR.glob("*.jsonl"))}
for name, rows in files.items():
    print(name, len(rows))

C:\Users\gregh\Desktop\OpenClassrooms\AI Engineer\P14\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


dpo_train 3000
dpo_val 300
eval_frenchmedmcqa 321
eval_mediqal_qcm 300
eval_mediqal_qcu 500
eval_ultramedical 777
sft_test 250
sft_train 4500
sft_val 250


## 1. Presidio avec les modèles spaCy français et anglais

Presidio a deux moteurs : l'`AnalyzerEngine` repère les entités sensibles, l'`AnonymizerEngine` les remplace. Pour reconnaître les noms de personnes, l'analyseur s'appuie sur un modèle de langue spaCy : `fr_core_news_md` pour le français, `en_core_web_md` pour l'anglais.

In [2]:
provider = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "fr", "model_name": "fr_core_news_md"},
               {"lang_code": "en", "model_name": "en_core_web_md"}],
})
nlp_engine = provider.create_engine()


def default_analyzer():
    registry = RecognizerRegistry(supported_languages=["fr", "en"])
    registry.load_predefined_recognizers(languages=["fr", "en"], nlp_engine=nlp_engine)
    return AnalyzerEngine(nlp_engine=nlp_engine, registry=registry, supported_languages=["fr", "en"])


analyzer = default_analyzer()
anonymizer = AnonymizerEngine()
ENTITIES = ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"]

## 2. Les stratégies de masquage

On compare les trois stratégies proposées par Presidio sur une phrase inventée.

In [3]:
example = "Monsieur Jean Dupont, 54 ans, joignable au 06 12 34 56 78 ou à jean.dupont@mail.fr, consulte pour une douleur thoracique."
found = analyzer.analyze(text=example, language="fr", entities=ENTITIES)
print([(r.entity_type, example[r.start:r.end]) for r in found])

strategies = {
    "replace": {"PERSON": OperatorConfig("replace", {"new_value": "<PERSONNE>"}),
                "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
                "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<TELEPHONE>"})},
    "mask": {"DEFAULT": OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 100, "from_end": False})},
    "redact": {"DEFAULT": OperatorConfig("redact", {})},
}
for name, operators in strategies.items():
    print(f"{name:8s}", anonymizer.anonymize(text=example, analyzer_results=found, operators=operators).text)

[('EMAIL_ADDRESS', 'jean.dupont@mail.fr'), ('PERSON', 'Jean Dupont'), ('PHONE_NUMBER', '06 12 34 56 78')]
replace  Monsieur <PERSONNE>, 54 ans, joignable au <TELEPHONE> ou à <EMAIL>, consulte pour une douleur thoracique.
mask     Monsieur ***********, 54 ans, joignable au ************** ou à *******************, consulte pour une douleur thoracique.
redact   Monsieur , 54 ans, joignable au  ou à , consulte pour une douleur thoracique.


On retient `replace`. La phrase reste lisible et on voit qu'une information a été retirée. `mask` garde la longueur exacte du nom, ce qui en dit un peu sur lui, et `redact` laisse des phrases trouées qui gêneraient l'apprentissage du modèle.

## 3. Premier essai : Presidio tel quel

On lance l'analyseur par défaut sur 300 exemples français et 300 anglais tirés au hasard, et on regarde ce qu'il prend pour des noms de personnes.

In [4]:
def texts_of(row):
    if "messages" in row:
        return [m["content"] for m in row["messages"]]
    return [row["prompt"][0]["content"], row["chosen"][0]["content"], row["rejected"][0]["content"]]


rng = random.Random(SEED)
pool = files["sft_train"] + files["dpo_train"]
sample = rng.sample([r for r in pool if r["langue"] == "fr"], 300) + rng.sample([r for r in pool if r["langue"] == "en"], 300)

detected = Counter()
for row in sample:
    for text in texts_of(row):
        for r in analyzer.analyze(text=text, language=row["langue"], entities=ENTITIES):
            detected[(r.entity_type, text[r.start:r.end])] += 1
print(sum(detected.values()), "détections sur 600 exemples")
print(detected.most_common(40))

1080 détections sur 600 exemples
[(('PERSON', 'Paget'), 17), (('PERSON', 'E.'), 13), (('PERSON', 'Alagille'), 12), (('PERSON', 'Becker'), 9), (('PERSON', 'Duane'), 8), (('PERSON', 'Vestibular Schwannoma'), 8), (('PERSON', 'Gardner'), 8), (('PERSON', 'Gaucher'), 7), (('PERSON', 'Vi'), 7), (('PERSON', 'Pallister-Killian'), 7), (('PERSON', 'Mainzer'), 7), (('PERSON', 'Anderson'), 6), (('PERSON', 'A. Serum'), 6), (('PERSON', 'Treg'), 6), (('PERSON', 'Giloy'), 6), (('PERSON', 'Kallmann'), 6), (('PERSON', 'D.'), 5), (('PERSON', 'Gustillo'), 5), (('PERSON', 'Brihatvata Chintamani Ras'), 5), (('PERSON', 'Monsieur'), 4), (('PERSON', 'E. A'), 4), (('PERSON', 'E. Est'), 4), (('PERSON', 'L4'), 4), (('PERSON', 'ilio-lombaire'), 4), (('PERSON', 'I.T.T. supérieure'), 4), (('PERSON', 'adnexa'), 4), (('PERSON', 'Chandler'), 4), (('PERSON', 'B. Diminution'), 3), (('PERSON', 'Madame Anne G.'), 3), (('PERSON', 'acétylsalicylate'), 3), (('PERSON', 'D. Est'), 3), (('PERSON', "D. N'"), 3), (('PERSON', 'Mme B

Constat : la plupart des « personnes » trouvées n'en sont pas. On y voit des lettres de réponses de QCM (« E. », « A. Est »), des médicaments, des bactéries, et surtout des maladies qui portent le nom d'un médecin (Cushing, Paget, Willebrand, Barrett). Le détecteur de téléphone, lui, prend aussi des tensions artérielles (« 168/98 ») et des numérations de plaquettes pour des numéros.

Masquer tout cela retirerait du sens médical aux textes. On écrit donc des règles plus fines.

## 4. Les règles retenues

On garde Presidio et ses deux moteurs, mais on change ce qui est détecté et ce qui est masqué :
1. Un nom qui suit une civilité (« Monsieur B. », « Mme P. D. », « Mrs. J. », « Dr S. »), une présentation (« je m'appelle », « my name is ») ou une salutation (« Hello P. », « Dear D. ») est masqué. La civilité elle-même est gardée : le sexe du patient compte pour le raisonnement.
2. Un nom repéré par spaCy n'est masqué que s'il contient un prénom connu. Les listes viennent de la bibliothèque Faker (prénoms français, américains, britanniques et indiens). Un nom de famille seul n'est pas masqué : en médecine, c'est presque toujours une maladie ou un signe (Cushing, Paget).
3. Un nom n'est jamais masqué s'il fait partie d'un nom de maladie, de test ou de structure : suivi de « syndrome », « disease », « test », « maladie », « signe »…, précédé de « maladie de », « facteur »…, ou écrit avec des traits d'union (Charcot-Marie-Tooth, Bernard-Soulier).
4. Quelques prénoms qui sont aussi des mots courants ou des noms de maladies ne comptent pas comme prénoms (Gene, Grace, Hope, Will, Duane, Douglas…). Ils ont été trouvés en relisant les premiers résultats.
5. Toutes les adresses e-mail sont masquées. Pour les téléphones, le détecteur fourni est remplacé par des formats stricts : numéros français (06 12 34 56 78, +33…) et américains (404-639-1075, 1-800-…).

Deux règles ont été ajoutées après un premier contrôle manuel (détails en partie 6) :
6. Une détection de spaCy qui commence par une civilité (« Mme M. », « Monsieur B. A. ») est masquée, civilité exceptée. Presidio ne garde que la plus grande de deux détections qui se chevauchent : la détection « M. » de la règle 1 disparaissait derrière « Mme M. » de spaCy. Les civilités en minuscules (« madame F. ») sont aussi reconnues.
7. Un nom suivi d'un âge (« A., âgée de 68 ans », « E., a 34-year-old ») est masqué, sauf les mots comme « Homme » ou « Patient ».

Au moment de masquer, deux ajustements viennent du troisième contrôle : après une civilité, on s'arrête à la fin du nom (« Monsieur B. urine énormément » ne doit pas perdre « urine énormément ») ; et si le nom continue, on prolonge le masque d'un ou deux mots en majuscule ou initiales (« My name is E. L. », « Dr. S.K. C. »), sauf « Il », « Elle », « The »… Enfin, une règle de rattrapage passe après Presidio : après « my name is » ou « je m'appelle », le mot suivant est masqué même s'il est écrit en minuscules (« my name is j. b. »), sauf « not », « and »…

In [5]:
def first_names(*providers):
    names = set()
    for p in providers:
        for attr in ("first_names", "first_names_male", "first_names_female"):
            names |= set(getattr(p, attr, ()) or ())
    return names


FIRST_NAMES = first_names(NamesFR, NamesUS, NamesGB, NamesIN)
NOT_FIRST_NAMES = {"Gene", "Grace", "Hope", "Will", "May", "Ray", "Joy", "Faith", "Chase", "Hunter", "Rose", "Mark",
                   "Bill", "Art", "Sage", "Sky", "Summer", "Autumn", "Dawn", "Iris", "Ivy", "Lily", "Ruby", "Amber",
                   "Crystal", "Duane", "Douglas", "Luc", "Todd", "Gilbert", "Leigh", "Stuart", "Marcus", "Robin"}
NOT_NAMES = {"Can", "Be", "If", "According", "Is", "Are", "Do", "Does", "What", "How", "Why", "When", "Please", "Thank",
             "Thanks", "My", "The", "This", "Hello", "Hi", "Sir", "Doctor", "Dr", "Madam", "Mam", "Team", "There", "All",
             "Everyone", "Shoot", "Grief", "Quelle", "Quel", "Quels", "Le", "La", "Les", "I'm", "Im"}
NOT_PATIENTS = {"Homme", "Femme", "Patient", "Patiente", "Enfant", "Garçon", "Fille", "Nourrisson", "Bébé", "Man",
                "Woman", "Boy", "Girl", "Child", "Infant", "Baby", "Cas"}
print(len(FIRST_NAMES), "prénoms connus")

UP, LOW = "A-ZÀÂÄÇÉÈÊËÎÏÔÖÙÛÜ", "a-zàâäçéèêëîïôöùûüÿ'’\\-"
NAME = rf"(?:[{UP}]\.\s?[{UP}][{LOW}]+|[{UP}][{LOW}]+\s[{UP}]\.|[{UP}][{LOW}]+|[{UP}]\.)"
TITLES = {"fr": ["Monsieur", "Madame", "Mademoiselle", "monsieur", "madame", "mademoiselle", "M.", "Mme", "Mlle", "Melle",
                 "Mr", "Dr", "Dr.", "Docteur", "Pr", "Professeur"],
          "en": ["Mr", "Mr.", "Mrs", "Mrs.", "Ms", "Ms.", "Miss", "Dr", "Dr.", "Doctor", "Prof.", "Professor"]}
INTROS = {"fr": ["je m'appelle", "je m’appelle", "Je m'appelle", "Je m’appelle"], "en": ["my name is", "My name is"]}
GREETINGS = {"fr": ["Bonjour", "Bonjour,"], "en": ["Hello", "Hello,", "Hi", "Hi,", "Dear", "Dear,", "Hey", "Hey,"]}
PHONES = {"fr": r"(?<!\d)(?:\+33\s?|0)[1-9](?:[\s.\-]?\d{2}){4}(?!\d)",
          "en": r"(?<!\d)(?:\+?1[\s.\-])?\(?\d{3}\)?[\s.\-]\d{3}[\s.\-]\d{4}(?!\d)"}


def after(words):
    # le nom qui suit un des mots donnés (le mot lui-même n'est pas masqué)
    return "(?:" + "|".join(f"(?<=\\b{re.escape(w)} )" for w in words) + ")" + NAME


def custom_analyzer():
    registry = RecognizerRegistry(supported_languages=["fr", "en"])
    registry.load_predefined_recognizers(languages=["fr", "en"], nlp_engine=nlp_engine)
    flags = re.DOTALL | re.MULTILINE  # sans IGNORECASE : la majuscule du nom compte
    for lang in ("fr", "en"):
        registry.remove_recognizer("PhoneRecognizer", language=lang)
        registry.add_recognizer(PatternRecognizer(
            supported_entity="PERSON", supported_language=lang, name=f"nom_apres_{lang}", global_regex_flags=flags,
            patterns=[Pattern("civilite", after(TITLES[lang]), 0.95), Pattern("presentation", after(INTROS[lang]), 0.95),
                      Pattern("salutation", after(GREETINGS[lang]), 0.95)]))
        registry.add_recognizer(PatternRecognizer(
            supported_entity="PHONE_NUMBER", supported_language=lang, name=f"telephone_{lang}", global_regex_flags=flags,
            patterns=[Pattern("telephone", PHONES[lang], 0.95)]))
    return AnalyzerEngine(nlp_engine=nlp_engine, registry=registry, supported_languages=["fr", "en"])


analyzer = custom_analyzer()

1399 prénoms connus


In [6]:
TRIGGERS = (r"syndrome|disease|sign|test|reaction|factor|palsy|paralysis|anomaly|sequence|pupil|procedure|phenomenon|"
            r"criteria|classification|score|scale|maneuver|triad|law|cells?|nodes?|nodules?|bod(?:y|ies)|tumou?r|"
            r"lymphoma|sarcoma|virus|titration|maladie|signe|réaction|facteur|paralysie|triade|manœuvre|cellules?|tumeur|"
            r"atlas|ontology|database|regulation|expression|cluster|mapping|tree|knockout|insertion|rearrangement|"
            r"pathway|protein|receptor|complex|hospital|children's|university|institute|centre|center")
NEXT_WORD = re.compile(rf"^(?:'s|’s|s')?\s*(?:{TRIGGERS})\b", re.IGNORECASE)
PREV_WORDS = re.compile(r"(?:syndrome|maladie|signe|test|réaction|facteur|paralysie|triade|angine|disease|sign)\s+"
                        r"(?:de\s+|d['’]\s*|du\s+|of\s+)?$", re.IGNORECASE)
IN_SPAN = re.compile(rf"\b(?:{TRIGGERS})\b", re.IGNORECASE)
GREETING_BEFORE = re.compile(r"(?:Hello|Hi|Dear|Hey|Thanks|Thank you|Bonjour)[ ,]+$", re.IGNORECASE)
TITLE_START = re.compile(r"^(?:" + "|".join(re.escape(t) for t in TITLES["fr"] + TITLES["en"]) + r")\s+(\S+)")
AGE_AFTER = re.compile(r"^,?\s+(?:âgée?\b|\d{1,3}\s+ans\b|(?:an?\s+)?\d{1,3}[- ]year[- ]old)", re.IGNORECASE)


def decide(entity, recognizer, span, before, after_text):
    # renvoie (masquer ?, raison)
    if entity in ("EMAIL_ADDRESS", "PHONE_NUMBER"):
        return True, entity.lower()
    if entity != "PERSON":
        return False, "autre type"
    if recognizer.startswith("nom_apres"):
        if span.rstrip(".'’s") in NOT_NAMES:
            return False, "faux nom après civilité"
        return True, "civilité, présentation ou salutation"
    title = TITLE_START.match(span)
    if title and title.group(1).rstrip(".,'’s") not in NOT_NAMES:
        return True, "civilité, présentation ou salutation"
    if "-" in span:
        return False, "nom composé (maladie)"
    if NEXT_WORD.match(after_text) or PREV_WORDS.search(before) or IN_SPAN.search(span):
        return False, "nom de maladie, de test ou de structure"
    words = [w for w in re.findall(r"[^\W\d_]+", span) if len(w) > 1 and not w.isupper()]
    if not words:
        return False, "sigle ou lettre"
    if GREETING_BEFORE.search(before) and words[0] not in NOT_NAMES:
        return True, "civilité, présentation ou salutation"
    if AGE_AFTER.match(after_text) and words[0] not in NOT_PATIENTS | NOT_NAMES:
        return True, "nom suivi d'un âge"
    if any(w in FIRST_NAMES and w not in NOT_FIRST_NAMES for w in words):
        return True, "prénom connu"
    return False, "pas de prénom connu"


TITLE_WORDS = re.compile(r"(?:Monsieur|Madame|Mademoiselle|monsieur|madame|mademoiselle|Mme|Mlle|Melle|M\.|"
                         r"Mr\.?|Mrs\.?|Ms\.?|Miss|Dr\.?|Docteur|Doctor|Pr\.?|Professeur|Professor)\s+")
NAME_ONLY = re.compile(rf"{NAME}(?:\s?{NAME}){{0,2}}")
NEXT_TOKEN = re.compile(rf" ?([{UP}]\.|[{UP}][{LOW}]+)")  # une seule espace, jamais de retour à la ligne
STOP_NEXT = {"Il", "Elle", "Ils", "Elles", "Le", "La", "Les", "Un", "Une", "Ce", "Cet", "Cette", "On", "Son", "Sa", "Ses",
             "Et", "He", "She", "They", "The", "His", "Her", "It", "In", "At", "I", "Is", "Was", "And"}


def trim(text, start, end):
    # ne pas masquer la lettre de réponse (« A. »), une civilité en tête, ni ce qui passe à la ligne
    span = text[start:end]
    if "\n" in span:
        end = start + span.index("\n")
    start += re.match(r"(?:[A-E]\.\s+)?", text[start:end]).end()
    title = TITLE_WORDS.match(text, start, end)
    if title:
        # après une civilité, on ne masque que le nom : « Monsieur B. urine énormément » -> « B. »
        start = title.end()
        name = NAME_ONLY.match(text, start, end)
        if name:
            end = name.end()
    return start, end


def extend(text, end):
    # nom en plusieurs morceaux : « E. » -> « E. L. », « S. » -> « S.K. C. »
    for _ in range(2):
        m = NEXT_TOKEN.match(text, end)
        if not m or m.group(1).rstrip(".") in STOP_NEXT:
            break
        end = m.end()
    return end

## 5. Application à toutes les données

Chaque texte (consigne et réponses) passe dans l'analyseur, puis chaque détection est triée par les règles. On garde la trace de toutes les décisions pour les contrôles.

In [7]:
TAGS = {"fr": {"PERSON": "<PERSONNE>", "EMAIL_ADDRESS": "<EMAIL>", "PHONE_NUMBER": "<TELEPHONE>"},
        "en": {"PERSON": "<PERSON>", "EMAIL_ADDRESS": "<EMAIL>", "PHONE_NUMBER": "<PHONE>"}}
TAG = re.compile(r"<(?:PERSONNE|PERSON|EMAIL|TELEPHONE|PHONE)>")
decisions = []
extensions = []
caught = []

# rattrapage après Presidio : un nom écrit en minuscules après une présentation (« my name is j. b. »)
INTRO_ANY_CASE = re.compile(r"\b(my name is|je m['’]appelle)\s+([^\W\d_][\w'’\-]*)(\s+[^\W\d_][\w'’\-]*)?", re.IGNORECASE)
INTRO_STOP = {"not", "a", "an", "the", "and", "i", "im", "from", "age", "aged", "am", "with", "is", "was", "but", "so",
              "et", "je", "j", "de", "le", "la", "un", "une", "mais"}


def catch_intro(text, lang):
    def repl(m):
        first, second = m.group(2), m.group(3)
        if first.lower() in INTRO_STOP:
            return m.group(0)
        caught.append(m.group(0))
        keep_second = second if second and second.strip().lower() in INTRO_STOP else ""
        return f"{m.group(1)} {TAGS[lang]['PERSON']}{keep_second}"
    return INTRO_ANY_CASE.subn(repl, text)


def anonymize_text(text, lang, row_id, file):
    kept = []
    for r in analyzer.analyze(text=text, language=lang, entities=ENTITIES):
        span, before, after_text = text[r.start:r.end], text[max(0, r.start - 50):r.start], text[r.end:r.end + 50]
        mask, reason = decide(r.entity_type, r.recognition_metadata.get("recognizer_name", ""), span, before, after_text)
        decisions.append({"fichier": file, "id": row_id, "langue": lang, "entite": r.entity_type, "texte": span,
                          "contexte": (before + "[[" + span + "]]" + after_text).replace("\n", " "),
                          "masque": mask, "raison": reason})
        if mask:
            r.start, r.end = trim(text, r.start, r.end)
            if r.entity_type == "PERSON" and r.end > r.start:
                end = extend(text, r.end)
                if end > r.end:
                    extensions.append((text[r.start:r.end], text[r.start:end]))
                r.end = end
            if r.end > r.start:
                kept.append(r)
    operators = {e: OperatorConfig("replace", {"new_value": tag}) for e, tag in TAGS[lang].items()}
    new_text = anonymizer.anonymize(text=text, analyzer_results=kept, operators=operators).text
    new_text, _ = catch_intro(new_text, lang)
    # un même nom peut être repéré deux fois (spaCy et nos motifs), Presidio n'en remplace qu'un :
    # on compte les étiquettes ajoutées dans le texte, pas les détections
    return new_text, len(TAG.findall(new_text)) - len(TAG.findall(text))


anonymized = {}
for name, rows in files.items():
    out = []
    for row in rows:
        row = json.loads(json.dumps(row))  # copie
        n = 0
        if "messages" in row:
            for m in row["messages"]:
                m["content"], k = anonymize_text(m["content"], row["langue"], row["id"], name)
                n += k
        else:
            for field in ("prompt", "chosen", "rejected"):
                row[field][0]["content"], k = anonymize_text(row[field][0]["content"], row["langue"], row["id"], name)
                n += k
        row["nb_elements_masques"] = n
        out.append(row)
    anonymized[name] = out
    print(f"{name} : {sum(r['nb_elements_masques'] for r in out)} éléments masqués dans "
          f"{sum(r['nb_elements_masques'] > 0 for r in out)} exemples sur {len(out)}")

dpo_train : 256 éléments masqués dans 87 exemples sur 3000


dpo_val : 17 éléments masqués dans 7 exemples sur 300


eval_frenchmedmcqa : 0 éléments masqués dans 0 exemples sur 321


eval_mediqal_qcm : 47 éléments masqués dans 35 exemples sur 300


eval_mediqal_qcu : 76 éléments masqués dans 54 exemples sur 500


eval_ultramedical : 80 éléments masqués dans 29 exemples sur 777


sft_test : 23 éléments masqués dans 19 exemples sur 250


sft_train : 405 éléments masqués dans 278 exemples sur 4500


sft_val : 16 éléments masqués dans 12 exemples sur 250


In [8]:
dec = pd.DataFrame(decisions)
print(len(dec), "détections au total\n")
print(dec.groupby(["masque", "raison"]).size().sort_values(ascending=False).to_string())

18738 détections au total

masque  raison                                 
False   pas de prénom connu                        12667
        nom de maladie, de test ou de structure     2096
        sigle ou lettre                             1584
        nom composé (maladie)                       1141
True    civilité, présentation ou salutation         941
        prénom connu                                 182
False   faux nom après civilité                       63
True    nom suivi d'un âge                            50
        phone_number                                  11
        email_address                                  3


In [9]:
# les noms prolongés au moment du masquage : on vérifie qu'on ne mange pas de mots médicaux
# (seulement les initiales à l'affichage, pour ne pas recopier les noms dans le notebook)
def initials(span):
    return " ".join(w[0] + "." for w in span.split())


print(len(extensions), "masques prolongés, par exemple :")
for before, after in random.Random(SEED).sample(sorted(extensions), 15):
    print(f"  {initials(before):10} -> {initials(after)}")
added = [after[len(before):] for before, after in extensions]
print("prolongements qui contiennent un mot médical :", sum(bool(IN_SPAN.search(a)) for a in added))
print("\nrattrapages après une présentation :", len(caught))

86 masques prolongés, par exemple :
  V.         -> V. G.
  B.         -> B.
  A.         -> A.
  H.         -> H.
  G.         -> G. P.
  F.         -> F.
  B.         -> B. D.
  B.         -> B. M.
  S.         -> S. K.
  B.         -> B. H.
  S.         -> S. T.
  M.         -> M. K.
  A.         -> A.
  F.         -> F.
  F.         -> F.
prolongements qui contiennent un mot médical : 0

rattrapages après une présentation : 8


## 6. Contrôle qualité

Trois contrôles :
1. Ce qu'on a masqué était-il vraiment un nom ou un identifiant ? On tire 60 éléments masqués au hasard et on les annote à la main.
2. Qu'a-t-on laissé passer ? On tire 100 détections de spaCy que les règles ont gardées (raison « pas de prénom connu ») et on les annote à la main. Les noms réels qu'on y trouve donnent une idée de ce qui reste.
3. Une recherche automatique sur les fichiers finaux : plus aucune adresse e-mail ni aucun numéro au format téléphone.

Les échantillons sont exportés en CSV dans `data/controle/`, annotés dans un tableur (colonne `vrai_nom` : 1 si c'est un vrai identifiant, 0 sinon), puis relus ici.

Un premier tour de contrôle a été fait avec les règles 1 à 5 (fichiers `tour1_*_annote.csv`). Parmi 60 éléments masqués, 56 étaient de vrais identifiants : les 4 erreurs étaient « angine de Vincent », le complexe protéique « TOM », « I'm » après « Hello, » et « Ian » mis pour « I am ». Parmi 100 détections gardées, 4 étaient de vrais noms : « Mme M. » (la civilité avalée par la détection de spaCy), « madame F. » en minuscules, « A., âgée de 68 ans », et un auteur cité en référence. D'où les règles 6 et 7 et quelques mots ajoutés aux listes.

Le deuxième tour (fichiers `tour2_*_annote.csv`) donnait 57 vrais identifiants sur 60 éléments masqués, et 2 vrais noms sur 100 détections gardées, deux auteurs cités en référence. Mais en relançant le notebook, la répartition des décisions changeait légèrement d'une exécution à l'autre. Quand spaCy et une de nos règles repèrent exactement le même passage avec le même score, Presidio n'en garde qu'un, choisi au hasard : « Monsieur G. » était masqué ou non selon l'exécution. Le score de nos règles a été relevé à 0,95 (0,85 pour spaCy), et deux exécutions donnent maintenant les mêmes décisions. Le troisième tour, ci-dessous, est fait sur cette version, avec de nouveaux échantillons tirés après un tri des décisions pour que le tirage soit reproductible. Il a encore conduit aux deux ajustements de la fin de la partie 4, qui ne changent pas les décisions (donc pas les échantillons), seulement la longueur des masques.

In [10]:
# tri avant le tirage : l'ordre des détections de Presidio peut varier d'une exécution à l'autre
ordered = dec.sort_values(["fichier", "id", "texte", "contexte"]).reset_index(drop=True)
masked = ordered[ordered["masque"]].sample(60, random_state=SEED)
missed = ordered[ordered["raison"] == "pas de prénom connu"].sample(100, random_state=SEED)
for df, name in ((masked, "echantillon_masques"), (missed, "echantillon_gardes")):
    path = CTRL_DIR / f"{name}.csv"
    df[["fichier", "id", "langue", "entite", "texte", "contexte", "raison"]].assign(vrai_nom="").to_csv(path, index=False, encoding="utf-8")
    print(path, len(df))

..\data\controle\echantillon_masques.csv 60
..\data\controle\echantillon_gardes.csv 100


In [11]:
def read_annotations(name):
    path = CTRL_DIR / f"{name}_annote.csv"
    if not path.exists():
        print(path.name, ": pas encore annoté")
        return None
    df = pd.read_csv(path, encoding="utf-8")
    drawn = pd.read_csv(CTRL_DIR / f"{name}.csv", encoding="utf-8")
    same = (df["contexte"].values == drawn["contexte"].values).all()
    print(path.name, ":", len(df), "lignes,", int(df["vrai_nom"].sum()), "vrais identifiants",
          "| même échantillon que le tirage :", same)
    return df


ann_masked = read_annotations("echantillon_masques")
ann_missed = read_annotations("echantillon_gardes")
if ann_masked is not None:
    print(f"\nparmi les éléments masqués, part de vrais identifiants : {100 * ann_masked['vrai_nom'].mean():.0f} %")
if ann_missed is not None:
    print(f"parmi les détections gardées, part de vrais noms laissés : {100 * ann_missed['vrai_nom'].mean():.0f} %")
    print(ann_missed[ann_missed["vrai_nom"] == 1][["texte", "contexte"]].to_string(max_colwidth=90))

echantillon_masques_annote.csv : 60 lignes, 56 vrais identifiants | même échantillon que le tirage : True
echantillon_gardes_annote.csv : 100 lignes, 1 vrais identifiants | même échantillon que le tirage : True

parmi les éléments masqués, part de vrais identifiants : 93 %
parmi les détections gardées, part de vrais noms laissés : 1 %
           texte                                                                                   contexte
57  Dancey et al  y accepted definition for gigantomastia; however, [[Dancey et al]]. (2007) state that ...


In [12]:
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
left = Counter()
for name, rows in anonymized.items():
    for row in rows:
        for text in texts_of(row):
            left["e-mails"] += len(EMAIL.findall(text))
            left["téléphones FR"] += len(re.findall(PHONES["fr"], text))
            left["téléphones US"] += len(re.findall(PHONES["en"], text))
print(dict(left))

{'e-mails': 0, 'téléphones FR': 0, 'téléphones US': 0}


Limites connues, à garder en tête :
- Quelques mots non personnels sont masqués : une partie d'un nom d'attelle ou de signe (« Thomas' splint », « Herbert's pits »), le mot « primigeste » suivi d'un âge, « Mr » dans « Dear Mr. R. ». C'est une perte de sens, pas un risque pour la vie privée.
- Les auteurs cités en référence (« Zhang et al. ») ne sont pas masqués. Ce sont des informations publiques, pas des données de santé sur ces personnes.
- Un prénom absent des listes de Faker et qui n'est précédé ni d'une civilité, ni d'une salutation, ni suivi d'un âge peut passer. Les contrôles n'en ont plus trouvé, mais 100 détections relues ne prouvent pas qu'il n'y en a aucun.
- Les données d'origine contenaient déjà des adresses masquées par leurs auteurs (« YYYY@YYYY »), qu'on laisse telles quelles.

## 7. Enregistrement

Les fichiers anonymisés vont dans `data/processed/`, au même format que l'entrée, avec en plus le champ `nb_elements_masques`. Le journal des décisions d'anonymisation est enregistré à côté, sans le texte des éléments masqués (il contiendrait les données qu'on vient de retirer).

In [13]:
for name, rows in anonymized.items():
    with open(OUT_DIR / f"{name}.jsonl", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"{name}.jsonl : {len(rows)} lignes")

journal = json.load(open(IN_DIR / "journal_transformations.json", encoding="utf-8"))
journal["anonymisation"] = {
    "outil": "Presidio 2.2.366 + spaCy fr_core_news_md / en_core_web_md",
    "strategie": "replace",
    "decisions": dec.groupby(["masque", "raison"]).size().reset_index(name="nombre").to_dict(orient="records"),
}
with open(OUT_DIR / "journal_transformations.json", "w", encoding="utf-8") as f:
    json.dump(journal, f, ensure_ascii=False, indent=1)

dpo_train.jsonl : 3000 lignes
dpo_val.jsonl : 300 lignes
eval_frenchmedmcqa.jsonl : 321 lignes
eval_mediqal_qcm.jsonl : 300 lignes
eval_mediqal_qcu.jsonl : 500 lignes
eval_ultramedical.jsonl : 777 lignes
sft_test.jsonl : 250 lignes
sft_train.jsonl : 4500 lignes
sft_val.jsonl : 250 lignes


## 8. Publication sur Hugging Face

Le jeu est publié en privé avec sa fiche (`dataset/README.md`), puis la version est marquée par l'étiquette `v1.0`. Seuls les fichiers anonymisés et le journal partent, jamais `data/interim/` ni `data/controle/`. Il faut être connecté (`uv run hf auth login`).

In [14]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = f"{api.whoami()['name']}/medical-triage-poc-data"
api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=OUT_DIR, repo_id=repo_id, repo_type="dataset",
                  allow_patterns=["*.jsonl", "journal_transformations.json"], commit_message="Données version 1.0")
api.upload_file(path_or_fileobj=Path("../dataset/README.md"), path_in_repo="README.md", repo_id=repo_id,
                repo_type="dataset", commit_message="Fiche du jeu de données")
api.create_tag(repo_id, tag="v1.0", repo_type="dataset", exist_ok=True)
print("publié :", f"https://huggingface.co/datasets/{repo_id}")
print(sorted(api.list_repo_files(repo_id, repo_type="dataset")))

publié : https://huggingface.co/datasets/g-hostin/medical-triage-poc-data
['.gitattributes', 'README.md', 'dpo_train.jsonl', 'dpo_val.jsonl', 'eval_frenchmedmcqa.jsonl', 'eval_mediqal_qcm.jsonl', 'eval_mediqal_qcu.jsonl', 'eval_ultramedical.jsonl', 'journal_transformations.json', 'sft_test.jsonl', 'sft_train.jsonl', 'sft_val.jsonl']
